# Phase 1 — Macro Census Baseline (America), state level

**Goal:** build the big-picture demographic map of America — the skeleton every later
layer (genetics, history, ancient DNA) hangs on.

Pulls ACS 5-year table **B03002 (Hispanic or Latino Origin by Race)** for every state + DC
and maps it to the macro groups in [`../docs/macro_groups.md`](../docs/macro_groups.md).
Geographic altitude follows [`../docs/granularity.md`](../docs/granularity.md): **state first**.

**Why B03002:** it separates Hispanic/Latino origin from race, giving mutually exclusive
macro categories (OMB-1997 style) — the clean cut for a baseline.

> Requires outbound network access to `api.census.gov`. A free API key
> (https://api.census.gov/data/key_signup.html) is optional at this volume; set it as
> `CENSUS_API_KEY` in `.env` (gitignored). The standalone equivalent is
> [`../scripts/fetch_census_baseline.py`](../scripts/fetch_census_baseline.py).

In [ ]:
import os
import pandas as pd
import requests

YEAR = 2023  # ACS 5-year end year

# B03002 variable -> macro group (see ../docs/macro_groups.md)
MACRO_VARS = {
    "B03002_001E": "total",
    "B03002_003E": "white_nh",
    "B03002_004E": "black_nh",
    "B03002_005E": "amerindian_ak_nh",
    "B03002_006E": "asian_nh",
    "B03002_007E": "nhpi_nh",
    "B03002_008E": "some_other_nh",
    "B03002_009E": "two_or_more_nh",
    "B03002_012E": "hispanic_any_race",
}
SHARE_COLS = [v for v in MACRO_VARS.values() if v != "total"]

In [ ]:
url = f"https://api.census.gov/data/{YEAR}/acs/acs5"
params = {"get": "NAME," + ",".join(MACRO_VARS), "for": "state:*"}
if os.environ.get("CENSUS_API_KEY"):
    params["key"] = os.environ["CENSUS_API_KEY"]

resp = requests.get(url, params=params, timeout=60)
resp.raise_for_status()
rows = resp.json()
df = pd.DataFrame(rows[1:], columns=rows[0])
df.head()

In [ ]:
# Tidy: rename to macro labels, coerce to numeric, tag the scheme
df = df.rename(columns=MACRO_VARS).rename(columns={"NAME": "state", "state": "state_fips"})
for col in MACRO_VARS.values():
    df[col] = pd.to_numeric(df[col])
df["scheme"] = "omb_1997"

# Shares of each macro group within the state total
for col in SHARE_COLS:
    df[f"{col}_share"] = (df[col] / df["total"]).round(5)

df = df.sort_values("state").reset_index(drop=True)
df[["state", "total"] + SHARE_COLS].head()

In [ ]:
# National rollup — the first real macro picture
national = df[list(MACRO_VARS.values())].sum()
national_shares = (national[SHARE_COLS] / national["total"]).sort_values(ascending=False)
print(f"Total population (states + DC): {int(national['total']):,}")
national_shares.round(4)

In [ ]:
# Save the small derived extract (this file belongs in git; see ../data/README.md)
out = f"../data/tier1_census/acs{YEAR}_5yr_state_macro_race_ethnicity.csv"
df.to_csv(out, index=False)
print("wrote", out)

---
**Next:** log the source in [`../CITATIONS.md`](../CITATIONS.md), then move to the **metro
(CBSA)** cut — where distinct communities start to separate (see `../docs/granularity.md`).